#### Manual Setup for each tool

In [ ]:
import datetime
import logging
from zoneinfo import ZoneInfo
from opentelemetry import trace
from dotenv import load_dotenv

load_dotenv()

logger = logging.getLogger("datetime_testing") # supplies name of the current module
tracer = trace.get_tracer("datetime_testing") # supplies name of the current module

def datetime_tool() -> str:
    """
    Get the current date and time in Nova Scotia (Atlantic Time).
    Use this tool to resolve relative time expressions like 'today', 'tomorrow', or 'next month'.

    Returns:
        The datetime as a string.
    """
    with tracer.start_as_current_span(
        "execute_tool datetime_tool",
        attributes={
            "gen_ai.operation.name": "execute_tool",
            "gen_ai.tool.name": "datetime_tool",
        },
    ) as span:
        ns_tz = ZoneInfo("America/Halifax")
        ns_now = datetime.datetime.now(ns_tz)
        result = ns_now.strftime("%A, %B %d, %Y %I:%M %p %Z")

        # Attach output to the span for agent evaluation / inspection
        span.set_attribute("gen_ai.tool.call.result", result)

        logger.info(
            "Datetime tool invoked.",
            extra={
                "event": "tool_call",
                "tool_name": "datetime_tool",
                "tool_result": result,
            },
        )

        return result

In [10]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
import os

chat_client = OpenAIChatClient(
    base_url=os.getenv("FOUNDRY_OPENAI_BASE_URL"),
    api_key=os.getenv("FOUNDRY_API_KEY"),
    model=os.getenv("FOUNDRY_MODEL"),
)

agent = Agent(
    client=chat_client,
    name="Date Time Agent",
    description="An agent that provides the current date and time in Nova Scotia.",
    instructions="Use your tools to get the current date and time in Nova Scotia.",
    tools=[datetime_tool],
    context_providers=[],
)

In [11]:
# Suppress noisy logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("agent_framework").setLevel(logging.WARNING)

# Supper noise logs
logging.basicConfig(level=logging.DEBUG)
result = await agent.run("Get the current date and time in Nova Scotia.")
print(result)

INFO:datetime_testing:Datetime tool invoked.


The current date and time in Nova Scotia (Atlantic Time) is:

**Friday, November 06, 2026 – 9:16 AM AST**


#### Automatic Setup for each tool

The following approach wraps the boilerplate into a decorator; however, you will lose flexibilty if it is a multi-step tool (like search)

In [ ]:
import functools
import inspect

def traced_tool(func):
    """Decorator that automatically creates a tool-specific span and log entry."""
    tool_name = func.__name__  # e.g., "datetime_tool", "jokes_tool"
    if inspect.iscoroutinefunction(func):
        @functools.wraps(func)
        async def async_wrapper(*args, **kwargs):
            with tracer.start_as_current_span(
                f"execute_tool {tool_name}",
                attributes={
                    "gen_ai.operation.name": "execute_tool",
                    "gen_ai.tool.name": tool_name,
                },
            ) as span:
                logger.info(f"Tool {tool_name} invoked.", extra={"event": "tool_call", "tool_name": tool_name})
                result = await func(*args, **kwargs)
                span.set_attribute("gen_ai.tool.call.result", str(result))
                return result
        return async_wrapper

    @functools.wraps(func)
    def sync_wrapper(*args, **kwargs):
        with tracer.start_as_current_span(
            f"execute_tool {tool_name}",
            attributes={
                "gen_ai.operation.name": "execute_tool",
                "gen_ai.tool.name": tool_name,
            },
        ) as span:
            logger.info(f"Tool {tool_name} invoked.", extra={"event": "tool_call", "tool_name": tool_name})
            result = func(*args, **kwargs)
            span.set_attribute("gen_ai.tool.call.result", str(result))
            return result
    return sync_wrapper

@traced_tool
def datetime_tool() -> str:
    """
    Get the current date and time in Nova Scotia (Atlantic Time).
    Use this tool to resolve relative time expressions like 'today', 'tomorrow', or 'next month'.

    Returns:
        The datetime as a string.
    """
    ns_tz = ZoneInfo("America/Halifax")
    ns_now = datetime.datetime.now(ns_tz)
    result = ns_now.strftime("%A, %B %d, %Y %I:%M %p %Z")

    logger.info(
        "Datetime tool invoked.",
        extra={
            "event": "tool_call",
            "tool_name": "datetime_tool",
            "tool_result": result,
        },
    )

    return result
